# SylloGym — Baseline Evaluation

This notebook evaluates the **base model** (before GRPO training) on all SylloGym tasks.
Run this before and after training to measure the improvement.

**Setup:**
1. Start the SylloGym server: `PYTHONPATH=envs uvicorn syllogym_env.server.app:app --port 8000`
2. Run this notebook with the model you want to evaluate

**Metrics tracked:**
- Accuracy per task
- Format compliance rate
- Average reward per task
- Reasoning quality score

In [ ]:
# ============================================================
# Setup — clone repo and install dependencies
# ============================================================

import os, sys

REPO_NAME = "syllogym"
REPO_URL = "https://github.com/eliot-gtn/syllogym.git"

# Clone the repo if not already present
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone -b develop {REPO_URL} /content/{REPO_NAME}
else:
    !git -C /content/{REPO_NAME} pull

# Install syllogym_env as a package (picks up openenv-core + datasets as deps)
%pip install -q -e /content/{REPO_NAME}/envs/syllogym_env

# Install evaluation dependencies
%pip install -q transformers torch accelerate huggingface_hub matplotlib

print("Setup complete.")

In [ ]:
from syllogym_env import SylloGymEnv, SylloAction
from syllogym_env.server.syllogym_environment import TASK_REGISTRY

print(f'SylloGym tasks: {len(TASK_REGISTRY)}')
for t in TASK_REGISTRY:
    print(f'  {t["name"]:30s} difficulty={t["difficulty"]} type={t["task_type"]}')

In [ ]:
# ============================================================
# Configuration
# ============================================================

SYLLOGYM_URL = "http://localhost:8000"   # Local server
# SYLLOGYM_URL = "https://YOUR_HF_SPACE.hf.space"  # HF Space

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Change to compare models
N_SAMPLES_PER_TASK = 30                   # Examples to evaluate per task
TASKS_TO_EVAL = [t["name"] for t in TASK_REGISTRY]  # All tasks

print(f'Model: {MODEL_NAME}')
print(f'Tasks: {TASKS_TO_EVAL}')
print(f'Samples per task: {N_SAMPLES_PER_TASK}')

In [ ]:
# ============================================================
# Load model
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f'Model loaded: {MODEL_NAME}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ============================================================
# Prompt builder
# ============================================================

SYSTEM_PROMPT = """You are a legal reasoning expert. You will be given:
1. A legal RULE
2. FACTS of a case
3. A QUESTION

Apply the rule to the facts using deductive reasoning (syllogism):
- Major premise: the rule
- Minor premise: the facts  
- Conclusion: your answer

Respond in this exact format:
<reasoning>
[Your step-by-step analysis applying the rule to the facts]
</reasoning>
<answer>[Your answer]</answer>

Keep your reasoning concise (2-4 sentences)."""


def build_user_prompt(obs) -> str:
    valid = " | ".join(obs.valid_answers)
    return (
        f"RULE:\n{obs.rule}\n\n"
        f"FACTS:\n{obs.facts}\n\n"
        f"QUESTION: {obs.question}\n\n"
        f"Valid answers: {valid}"
    )


def generate_response(obs, max_new_tokens=512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(obs)},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def parse_action_from_response(response: str) -> SylloAction:
    """Extract reasoning and answer from the model's response."""
    import re
    reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
    answer_match = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL | re.IGNORECASE)
    
    reasoning = reasoning_match.group(1).strip() if reasoning_match else response
    answer = answer_match.group(1).strip() if answer_match else response.strip()
    
    return SylloAction(reasoning=reasoning, answer=answer)


print('Prompt builder ready.')

In [ ]:
# ============================================================
# Evaluation loop
# ============================================================

from collections import defaultdict

env = SylloGymEnv(base_url=SYLLOGYM_URL)

results_by_task = defaultdict(list)  # task_name -> list of {reward, correct, format}

for task_name in TASKS_TO_EVAL:
    print(f'\nEvaluating: {task_name}', flush=True)
    
    task_rewards = []
    task_correct = []
    task_format = []
    
    for i in range(N_SAMPLES_PER_TASK):
        # Reset to get a new example from this specific task
        result = env.reset(task_mode="single", task_name=task_name)
        obs = result.observation
        
        if not obs.facts or obs.facts.startswith("[Dataset unavailable"):
            print(f'  [skip] Dataset unavailable for {task_name}')
            break
        
        # Generate model response
        response = generate_response(obs)
        action = parse_action_from_response(response)
        
        # Submit and get reward
        step_result = env.step(action)
        obs_after = step_result.observation
        
        reward = obs_after.reward or 0.0
        correct = obs_after.metadata.get("answer_reward", 0.0) > 0.5
        has_format = obs_after.metadata.get("format_reward", 0.0) > 0
        
        task_rewards.append(reward)
        task_correct.append(correct)
        task_format.append(has_format)
        
        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{N_SAMPLES_PER_TASK}] acc={sum(task_correct)/len(task_correct):.1%} reward={sum(task_rewards)/len(task_rewards):.3f}')
    
    if task_rewards:
        results_by_task[task_name] = {
            "accuracy": sum(task_correct) / len(task_correct),
            "avg_reward": sum(task_rewards) / len(task_rewards),
            "format_rate": sum(task_format) / len(task_format),
            "n": len(task_rewards),
        }
        print(f'  -> accuracy={results_by_task[task_name]["accuracy"]:.1%} '
              f'avg_reward={results_by_task[task_name]["avg_reward"]:.3f} '
              f'format={results_by_task[task_name]["format_rate"]:.1%}')

env.close()
print('\nEvaluation complete.')

In [ ]:
# ============================================================
# Summary table
# ============================================================

import json

print(f'\n{"="*70}')
print(f'BASELINE RESULTS — {MODEL_NAME}')
print(f'{"="*70}')
print(f'{"Task":<30} {"Acc":>8} {"Reward":>8} {"Format":>8} {"N":>5}')
print(f'{"-"*70}')

all_acc = []
all_reward = []
for task_name, r in sorted(results_by_task.items()):
    print(f'{task_name:<30} {r["accuracy"]:>8.1%} {r["avg_reward"]:>8.3f} {r["format_rate"]:>8.1%} {r["n"]:>5}')
    all_acc.append(r["accuracy"])
    all_reward.append(r["avg_reward"])

print(f'{"-"*70}')
if all_acc:
    print(f'{"OVERALL":<30} {sum(all_acc)/len(all_acc):>8.1%} {sum(all_reward)/len(all_reward):>8.3f}')

# Save results for comparison
output_path = f'baseline_{MODEL_NAME.replace("/", "_")}.json'
with open(output_path, 'w') as f:
    json.dump({
        "model": MODEL_NAME,
        "results": dict(results_by_task)
    }, f, indent=2)
print(f'\nSaved to {output_path}')

In [ ]:
# ============================================================
# Visualize results
# ============================================================

try:
    import matplotlib.pyplot as plt
    import numpy as np

    tasks = list(results_by_task.keys())
    accs = [results_by_task[t]["accuracy"] for t in tasks]
    rewards = [results_by_task[t]["avg_reward"] for t in tasks]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    colors = plt.cm.RdYlGn([a for a in accs])  # Red to Green by accuracy

    ax1.barh(tasks, accs, color=colors)
    ax1.set_xlabel('Accuracy')
    ax1.set_title(f'Task Accuracy — {MODEL_NAME.split("/")[-1]} (baseline)')
    ax1.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='random')
    ax1.set_xlim(0, 1)
    ax1.legend()

    ax2.barh(tasks, rewards, color=colors)
    ax2.set_xlabel('Average Reward')
    ax2.set_title(f'Average Reward — {MODEL_NAME.split("/")[-1]} (baseline)')
    ax2.set_xlim(0, 1.3)

    plt.tight_layout()
    plt.savefig(f'baseline_{MODEL_NAME.replace("/", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved.')
except ImportError:
    print('matplotlib not available — skipping chart')